In [55]:
x = torch.randn(3,3,2 ) * 0.1
print(x)
print()
print(x[:,0])

tensor([[[-0.1194,  0.0725],
         [ 0.0843, -0.1512],
         [-0.0079,  0.0138]],

        [[ 0.0666, -0.0726],
         [ 0.1641, -0.0283],
         [ 0.0749, -0.0284]],

        [[-0.0545, -0.0751],
         [-0.0365, -0.0491],
         [-0.0684, -0.0680]]])

tensor([[-0.1194,  0.0725],
        [ 0.0666, -0.0726],
        [-0.0545, -0.0751]])


In [56]:
import torch
import torch.nn as nn


class MPS(nn.Module):
    def __init__(self, num_sites, physical_dim=2, bond_dim=3):
        super().__init__()

        if num_sites < 2:
            raise ValueError("num_sites must be at least 2")

        self.num_sites = num_sites
        self.physical_dim = physical_dim
        self.bond_dim = bond_dim

        tensors = []

        # First site: no left bond
        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim) * 0.1
            )
        )

        # Middle sites: left and right bonds
        for _ in range(num_sites - 2):
            tensors.append(
                nn.Parameter(
                    torch.randn(physical_dim, bond_dim, bond_dim) * 0.1
                )
            )

        # Final site: no right bond
        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim) * 0.1
            )
        )

        self.tensors = nn.ParameterList(tensors)

    def forward(self, x):
        """
        x shape: [batch_size, num_sites, physical_dim]
        """

        if x.ndim != 3:
            raise ValueError(
                "x must have shape [batch_size, num_sites, physical_dim]"
            )

        if x.shape[1] != self.num_sites:
            raise ValueError(
                f"Expected {self.num_sites} sites, got {x.shape[1]}"
            )

        if x.shape[2] != self.physical_dim:
            raise ValueError(
                f"Expected physical_dim={self.physical_dim}, got {x.shape[2]}"
            )

        # First site:
        # [batch, physical_dim] × [physical_dim, bond_dim]
        # → [batch, bond_dim]
        # input of first site for each batch x 
        state = x[:, 0] @ self.tensors[0]

        # Middle sites:
        # Contract the physical input and the incoming bond.
        
        # indices:
        # b - runs through Batch
        # k - runs through inputs
        # d - tensor 
        for site in range(1, self.num_sites - 1):
            state = torch.einsum(
                "bl,blp,lpr->br",
                state,
                x[:, site],
                self.tensors[site]
            )

        # Final site:
        # [batch, bond_dim] contracted with the final physical input
        # and the final MPS tensor → [batch]
        y = torch.einsum(
            "bl,bp,pl->b",
            state,
            x[:, -1],
            self.tensors[-1]
        )

        return y

        

In [ ]:
model = MPS(num_sites=2,physical_dim=1, bond_dim=3)

x = torch.tensor([
    [[1.0], [0.0]],
    [[1.0], [2.0]],
    [[5.0], [8.0]],
    [[7.0], [4.0]],
    [[1.0], [1.0]],
    [[1.0], [1.0]],
    [[9.0], [0.0]]
])


target = torch.tensor([0,2,40,28,1,0])  

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

for epoch in range(100):
    pred = model.forward(x)
    loss = torch.mean((pred - target) ** 2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(model(x))

tensor([ 0.0000,  1.9734, 39.4671, 27.6269,  0.9867,  0.0000],
       grad_fn=<ViewBackward0>)
